In [4]:
import pandas as pd

file_path = "/content/online_retail_II.xlsx"

sheets = pd.read_excel(file_path, sheet_name=None)

for sheet_name, df in sheets.items():
    print("\nSheet:", sheet_name)
    print(df.columns.tolist())


Sheet: Year 2009-2010
['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']

Sheet: Year 2010-2011
['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country', 'Column1']


In [6]:

import pandas as pd
import numpy as np

# 1. Import the Excel workbook
file_path = "/content/online_retail_II.xlsx"

sheets = pd.read_excel(file_path, sheet_name=None)

processed_sheets = {}

# 2. Preprocess both sheets
for sheet_name, df in sheets.items():

    print(f"\nProcessing sheet: {sheet_name}")

    # Clean column names
    df.columns = (
        df.columns.astype(str)
        .str.strip()
        .str.replace(" ", "", regex=False)
    )

    # Standardize column names
    df.rename(columns={
        "Invoice": "InvoiceNo",
        "Price": "UnitPrice"
    }, inplace=True)

    # Check required columns
    required_columns = [
        "InvoiceNo", "StockCode", "Description",
        "Quantity", "InvoiceDate", "UnitPrice",
        "CustomerID", "Country"
    ]

    missing_columns = [
        col for col in required_columns
        if col not in df.columns
    ]

    if missing_columns:
        raise ValueError(
            f"Missing columns in {sheet_name}: {missing_columns}"
        )

    # Remove duplicate records
    original_rows = len(df)
    df.drop_duplicates(inplace=True)
    duplicates_removed = original_rows - len(df)

    # Clean text columns
    df["Description"] = df["Description"].astype("string").str.strip()
    df["Country"] = df["Country"].astype("string").str.strip()

    # Convert columns to appropriate data types
    df["InvoiceDate"] = pd.to_datetime(
        df["InvoiceDate"], errors="coerce"
    )

    df["Quantity"] = pd.to_numeric(
        df["Quantity"], errors="coerce"
    )

    df["UnitPrice"] = pd.to_numeric(
        df["UnitPrice"], errors="coerce"
    )

    # Identify cancelled invoices
    df["IsCancelled"] = (
        df["InvoiceNo"]
        .astype("string")
        .str.startswith("C", na=False)
    )

    # Identify product returns
    df["IsReturn"] = df["Quantity"] < 0

    # Identify zero or negative prices
    df["IsInvalidPrice"] = df["UnitPrice"] <= 0

    # Calculate revenue
    # Exclude cancelled invoices and invalid prices
    # Preserve negative revenue for product returns
    valid_revenue = (
        (~df["IsCancelled"]) &
        (df["UnitPrice"] > 0) &
        df["Quantity"].notna()
    )

    df["Revenue"] = np.where(
        valid_revenue,
        df["Quantity"] * df["UnitPrice"],
        0
    )

    # Extract year and month
    df["Year"] = df["InvoiceDate"].dt.year
    df["Month"] = df["InvoiceDate"].dt.month

    # Store processed sheet
    processed_sheets[sheet_name] = df

    # Display preprocessing summary
    print("Original records:", original_rows)
    print("Duplicates removed:", duplicates_removed)
    print("Final records:", len(df))
    print("Cancelled invoices:", df["IsCancelled"].sum())
    print("Product returns:", df["IsReturn"].sum())
    print("Invalid prices:", df["IsInvalidPrice"].sum())
    print("Missing values:", df.isnull().sum().sum())

# 3. Export both sheets into one Excel workbook
with pd.ExcelWriter("preprocessed.xlsx") as writer:

    for sheet_name, df in processed_sheets.items():

        df.to_excel(
            writer,
            sheet_name=sheet_name[:31],
            index=False
        )

print("\n================================")
print("PREPROCESSING COMPLETED!")
print("Output file: preprocessed.xlsx")
print("================================")



Processing sheet: Year 2009-2010
Original records: 525461
Duplicates removed: 6865
Final records: 518596
Cancelled invoices: 10182
Product returns: 12302
Invalid prices: 3684
Missing values: 110761

Processing sheet: Year 2010-2011
Original records: 541910
Duplicates removed: 5268
Final records: 536642
Cancelled invoices: 9251
Product returns: 10587
Invalid prices: 2512
Missing values: 673133

PREPROCESSING COMPLETED!
Output file: preprocessed.xlsx


In [7]:
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Column1,IsCancelled,IsReturn,IsInvalidPrice,Revenue,Year,Month
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,NaN,False,False,False,15.30,2010,12
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,NaN,False,False,False,20.34,2010,12
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,NaN,False,False,False,22.00,2010,12
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,NaN,False,False,False,20.34,2010,12
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,NaN,False,False,False,20.34,2010,12


In [8]:
print("Missing StockCodes:", df["StockCode"].isna().sum())

print("Unique StockCodes:", df["StockCode"].nunique())

print("StockCode data types:")
print(df["StockCode"].apply(type).value_counts())

print("Non-numeric StockCodes:")
print(
    df.loc[
        ~df["StockCode"].astype("string").str.fullmatch(r"\d+"),
        "StockCode"
    ].value_counts()
)

Missing StockCodes: 0
Unique StockCodes: 4070
StockCode data types:
StockCode
<class 'int'>    482154
<class 'str'>     54488
Name: count, dtype: int64
Non-numeric StockCodes:
StockCode
85123A     2301
85099B     2156
POST       1257
85099C      957
82494L      939
           ... 
51014c        1
47593b        1
79323GR       1
79323LP       1
79323G        1
Name: count, Length: 1124, dtype: int64


In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 536642 entries, 0 to 541909
Data columns (total 15 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   InvoiceNo       536642 non-null  object        
 1   StockCode       536642 non-null  object        
 2   Description     535188 non-null  string        
 3   Quantity        536642 non-null  int64         
 4   InvoiceDate     536642 non-null  datetime64[ns]
 5   UnitPrice       536642 non-null  float64       
 6   CustomerID      401605 non-null  float64       
 7   Country         536642 non-null  string        
 8   Column1         0 non-null       float64       
 9   IsCancelled     536642 non-null  boolean       
 10  IsReturn        536642 non-null  bool          
 11  IsInvalidPrice  536642 non-null  bool          
 12  Revenue         536642 non-null  float64       
 13  Year            536642 non-null  int32         
 14  Month           536642 non-null  int32   

In [13]:
zero_price_count = len(df[df["UnitPrice"] == 0])
print(zero_price_count)

2510


In [14]:

import pandas as pd

file_path = "/content/preprocessed.xlsx"

sheets = pd.read_excel(file_path, sheet_name=None)

for sheet_name, df in sheets.items():

    if sheet_name == "Data_Quality_Report":
        continue

    print("\n==========", sheet_name, "==========")

    # Separate zero-price records
    zero_price = df[df["UnitPrice"] == 0]

    # Missing CustomerID among zero-price records
    missing_id_zero_price = zero_price["CustomerID"].isna().sum()

    # Non-missing CustomerID among zero-price records
    valid_id_zero_price = zero_price["CustomerID"].notna().sum()

    # Missing CustomerID with non-zero price
    missing_id_nonzero_price = (
        df["CustomerID"].isna() &
        (df["UnitPrice"] != 0)
    ).sum()

    print("Total zero-price records:", len(zero_price))
    print("Zero price + missing CustomerID:", missing_id_zero_price)
    print("Zero price + existing CustomerID:", valid_id_zero_price)
    print("Missing CustomerID + non-zero price:", missing_id_nonzero_price)

    print(
        "\nUnique customers with zero-price transactions:",
        zero_price["CustomerID"].nunique()
    )

    print("\nCustomerID and UnitPrice combinations:")
    print(
        pd.crosstab(
            df["CustomerID"].isna(),
            df["UnitPrice"] == 0,
            margins=True
        )
    )



========== Year 2009-2010 ==========
Total zero-price records: 3681
Zero price + missing CustomerID: 3650
Zero price + existing CustomerID: 31
Missing CustomerID + non-zero price: 104183

Unique customers with zero-price transactions: 24

CustomerID and UnitPrice combinations:
UnitPrice    False  True     All
CustomerID                      
False       410732    31  410763
True        104183  3650  107833
All         514915  3681  518596

========== Year 2010-2011 ==========
Total zero-price records: 2510
Zero price + missing CustomerID: 2470
Zero price + existing CustomerID: 40
Missing CustomerID + non-zero price: 132567

Unique customers with zero-price transactions: 31

CustomerID and UnitPrice combinations:
UnitPrice    False  True     All
CustomerID                      
False       401565    40  401605
True        132567  2470  135037
All         534132  2510  536642


In [18]:
print(df.isnull().sum())

InvoiceNo              0
StockCode              0
Description         1454
Quantity               0
InvoiceDate            0
UnitPrice              0
CustomerID        135037
Country                0
Column1           536642
IsCancelled            0
IsReturn               0
IsInvalidPrice         0
Revenue                0
Year                   0
Month                  0
dtype: int64


In [19]:
import pandas as pd
import numpy as np

file_path = "/content/preprocessed.xlsx"

sheets = pd.read_excel(file_path, sheet_name=None)

for sheet_name, df in sheets.items():

    print("\n" + "=" * 50)
    print("SHEET:", sheet_name)
    print("=" * 50)

    print("\nShape:")
    print(df.shape)

    print("\nColumn Names:")
    print(df.columns.tolist())

    print("\nData Types:")
    print(df.dtypes)

    print("\nMissing Values:")
    print(df.isnull().sum())

    print("\nDuplicate Records:")
    print(df.duplicated().sum())

    print("\nFirst 5 Records:")
    print(df.head())


SHEET: Year 2009-2010

Shape:
(518596, 14)

Column Names:
['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country', 'IsCancelled', 'IsReturn', 'IsInvalidPrice', 'Revenue', 'Year', 'Month']

Data Types:
InvoiceNo                 object
StockCode                 object
Description               object
Quantity                   int64
InvoiceDate       datetime64[ns]
UnitPrice                float64
CustomerID               float64
Country                   object
IsCancelled                 bool
IsReturn                    bool
IsInvalidPrice              bool
Revenue                  float64
Year                       int64
Month                      int64
dtype: object

Missing Values:
InvoiceNo              0
StockCode              0
Description         2928
Quantity               0
InvoiceDate            0
UnitPrice              0
CustomerID        107833
Country                0
IsCancelled            0
IsReturn               0
IsInv

In [20]:

import pandas as pd

file_path = "/content/preprocessed.xlsx"
sheets = pd.read_excel(file_path, sheet_name=None)

for sheet_name, df in sheets.items():

    print("\n" + "=" * 50)
    print("SHEET:", sheet_name)
    print("=" * 50)

    print("Zero UnitPrice:", (df["UnitPrice"] == 0).sum())
    print("Negative UnitPrice:", (df["UnitPrice"] < 0).sum())
    print("Positive UnitPrice:", (df["UnitPrice"] > 0).sum())

    print("Zero Quantity:", (df["Quantity"] == 0).sum())
    print("Negative Quantity:", (df["Quantity"] < 0).sum())
    print("Positive Quantity:", (df["Quantity"] > 0).sum())

    print("Cancelled invoices:", df["IsCancelled"].sum())

    print("\nMissing CustomerID:", df["CustomerID"].isna().sum())
    print("Missing Description:", df["Description"].isna().sum())

    print("\nNegative UnitPrice examples:")
    print(
        df.loc[df["UnitPrice"] < 0,
               ["InvoiceNo", "StockCode", "Description",
                "Quantity", "UnitPrice", "CustomerID"]]
        .head(10)
    )

    print("\nZero Quantity examples:")
    print(
        df.loc[df["Quantity"] == 0,
               ["InvoiceNo", "StockCode", "Description",
                "Quantity", "UnitPrice", "CustomerID"]]
        .head(10)
    )



SHEET: Year 2009-2010
Zero UnitPrice: 3681
Negative UnitPrice: 3
Positive UnitPrice: 514912
Zero Quantity: 0
Negative Quantity: 12302
Positive Quantity: 506294
Cancelled invoices: 10182

Missing CustomerID: 107833
Missing Description: 2928

Negative UnitPrice examples:
       InvoiceNo StockCode      Description  Quantity  UnitPrice  CustomerID
177338   A506401         B  Adjust bad debt         1  -53594.36         NaN
273025   A516228         B  Adjust bad debt         1  -44031.79         NaN
398731   A528059         B  Adjust bad debt         1  -38925.87         NaN

Zero Quantity examples:
Empty DataFrame
Columns: [InvoiceNo, StockCode, Description, Quantity, UnitPrice, CustomerID]
Index: []

SHEET: Year 2010-2011
Zero UnitPrice: 2510
Negative UnitPrice: 2
Positive UnitPrice: 534130
Zero Quantity: 0
Negative Quantity: 10587
Positive Quantity: 526055
Cancelled invoices: 9251

Missing CustomerID: 135037
Missing Description: 1454

Negative UnitPrice examples:
       InvoiceNo Stock

In [21]:

import pandas as pd

file_path = "/content/preprocessed.xlsx"
sheets = pd.read_excel(file_path, sheet_name=None)

for sheet_name, df in sheets.items():

    print("\n==========", sheet_name, "==========")

    # Negative quantity records
    returns = df[df["Quantity"] < 0]

    print("\nNegative Quantity - Invoice Type:")
    print(returns["IsCancelled"].value_counts())

    print("\nTop negative quantity products:")
    print(
        returns.groupby(
            ["StockCode", "Description"], dropna=False
        ).size().sort_values(ascending=False).head(15)
    )

    # Zero-price records
    zero_price = df[df["UnitPrice"] == 0]

    print("\nTop zero-price products:")
    print(
        zero_price.groupby(
            ["StockCode", "Description"], dropna=False
        ).size().sort_values(ascending=False).head(15)
    )

    print("\nZero-price sample:")
    print(
        zero_price[
            ["InvoiceNo", "StockCode", "Description",
             "Quantity", "UnitPrice", "CustomerID", "Country"]
        ].head(10).to_string(index=False)
    )



========== Year 2009-2010 ==========

Negative Quantity - Invoice Type:
IsCancelled
True     10181
False     2121
Name: count, dtype: int64

Top negative quantity products:
StockCode  Description                        
M          Manual                                 292
22138      BAKING SET 9 PIECE RETROSPOT           184
22423      REGENCY CAKESTAND 3 TIER               166
21232      STRAWBERRY CERAMIC TRINKET BOX         128
79323W     WHITE CHERRY LIGHTS                    125
POST       POSTAGE                                103
D          Discount                                95
85123A     WHITE HANGING HEART T-LIGHT HOLDER      93
79323P     PINK CHERRY LIGHTS                      84
21843      RED RETROSPOT CAKE STAND                69
79323LP    LIGHT PINK CHERRY LIGHTS                60
85042      ANTIQUE LILY FAIRY LIGHTS               59
21527      RED RETROSPOT TRADITIONAL TEAPOT        53
22617      BAKING SET SPACEBOY DESIGN              53
22061      LARGE CAKE S

In [22]:

import pandas as pd
import numpy as np

file_path = "/content/preprocessed.xlsx"
sheets = pd.read_excel(file_path, sheet_name=None)

for sheet_name, df in sheets.items():

    print("\n==========", sheet_name, "==========")

    expected_revenue = np.where(
        (~df["IsCancelled"]) &
        (df["UnitPrice"] > 0),
        df["Quantity"] * df["UnitPrice"],
        0
    )

    difference = np.abs(df["Revenue"] - expected_revenue)

    print("Revenue mismatches:", (difference > 0.001).sum())

    print("Existing total revenue:", df["Revenue"].sum())

    print("Expected total revenue:", expected_revenue.sum())

    print("Cancelled invoice revenue:",
          df.loc[df["IsCancelled"], "Revenue"].sum())

    print("Negative price revenue:",
          df.loc[df["UnitPrice"] < 0, "Revenue"].sum())



========== Year 2009-2010 ==========
Revenue mismatches: 0
Existing total revenue: 10271762.664
Expected total revenue: 10271762.664
Cancelled invoice revenue: 0.0
Negative price revenue: 0.0

========== Year 2010-2011 ==========
Revenue mismatches: 0
Existing total revenue: 10642128.804
Expected total revenue: 10642128.804
Cancelled invoice revenue: 0.0
Negative price revenue: 0.0


In [23]:

import pandas as pd

input_file = "/content/preprocessed.xlsx"
output_file = "/content/RetailIQ_Final.xlsx"

sheets = pd.read_excel(input_file, sheet_name=None)

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:

    for sheet_name, df in sheets.items():

        print("\nProcessing:", sheet_name)

        # Remove completely empty columns
        df = df.dropna(axis=1, how="all")

        # Remove exact duplicate records
        df = df.drop_duplicates()

        # Save each year in a separate sheet
        df.to_excel(
            writer,
            sheet_name=sheet_name[:31],
            index=False
        )

        print("Final shape:", df.shape)
        print("Missing CustomerID:", df["CustomerID"].isna().sum())
        print("Missing Description:", df["Description"].isna().sum())
        print("Duplicate records:", df.duplicated().sum())

print("\nFinal preprocessing completed!")
print("Saved file:", output_file)



Processing: Year 2009-2010
Final shape: (518596, 14)
Missing CustomerID: 107833
Missing Description: 2928
Duplicate records: 0

Processing: Year 2010-2011
Final shape: (536642, 14)
Missing CustomerID: 135037
Missing Description: 1454
Duplicate records: 0

Final preprocessing completed!
Saved file: /content/RetailIQ_Final.xlsx


In [24]:
final_sheets = pd.read_excel(
    "/content/RetailIQ_Final.xlsx",
    sheet_name=None
)

for name, df in final_sheets.items():
    print("\n", name)
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    print("Duplicates:", df.duplicated().sum())
    print("Revenue:", df["Revenue"].sum())



 Year 2009-2010
Shape: (518596, 14)
Columns: ['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country', 'IsCancelled', 'IsReturn', 'IsInvalidPrice', 'Revenue', 'Year', 'Month']
Duplicates: 0
Revenue: 10271762.664

 Year 2010-2011
Shape: (536642, 14)
Columns: ['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country', 'IsCancelled', 'IsReturn', 'IsInvalidPrice', 'Revenue', 'Year', 'Month']
Duplicates: 0
Revenue: 10642128.804
